# Pangaea Benchmark — Results Analysis
Load results from local experiment directories and build pivot tables.

In [ ]:
import re
import os
from pathlib import Path
import pandas as pd
import numpy as np

EXPERIMENTS_BASE = Path("/dss/dsstbyfs02/pn49cu/pn49cu-dss-0001/niklas/experiments")

# ── Display name mappings ────────────────────────────────────────────────────
ENCODER_LABELS = {
    "carl_no_internal_norm":    "CARL",
    "carl_imagewise":           "CARL-Imagewise",
    "dofa":                     "DOFA",
    "dofav2":                   "DOFAv2",
    "carl":                     "CARL",
    "panopticon":               "Panopticon",
    "copernicus_fm_base":       "Copernicus-FM",
    "smarties_vitb":            "SMARTIES",
    "dinov2_vitb14":            "DINOv2",
    "dinov3_vitl16_sat493m":    "DINOv3-ViT-L",
    "unet_encoder":             "UNet",
    "unet_encoder_mi":          "UNet",
    "vit_scratch":              "ViT-Scratch",
    "vit_scratch_mi":           "ViT-Scratch",
    "vit_scratch_p8":           "ViT-Scratch-p8",
    "vit_scratch_mi_p8":        "ViT-Scratch-p8",
}

DATASET_LABELS = {
    "hlsburnscars":      "HLS-BurnScars",
    "sen1floods11":      "Sen1Floods11",
    "mados":             "MADOS",
    "pastis":            "PASTIS",
    "dynamicen":         "DynamicEarthNet",
    "croptypemapping":   "CropTypeMapping",
    "spacenet7cd":       "SpaceNet7-CD",
    "ai4smallfarms":     "AI4SmallFarms",
    "fivebillionpixels": "FiveBillionPixels",
}

DATASET_ORDER = list(DATASET_LABELS.keys())

# Encoders listed in longest-first order for greedy name matching.
# carl_imagewise MUST appear before carl to avoid carl matching carl_imagewise_*.
KNOWN_ENCODERS = [
    "copernicus_fm_base", "copernicus_fm_large",
    "carl_no_internal_norm",
    "carl_imagewise",
    "dinov3_vitl16_sat493m",
    "smarties_vitb", "dinov2_vitb14",
    "dofav2", "dofa", "panopticon", "carl",
    "vit_scratch_mi_p8", "vit_scratch_p8",
    "vit_scratch_mi", "vit_scratch",
    "unet_encoder_mi", "unet_encoder",
]

KNOWN_DECODERS = [
    "seg_upernet_mt_ltae", "seg_upernet",
    "seg_siamupernet_conc", "seg_siamupernet_diff",
    "seg_siam_mlp_fusion", "seg_mlp_fusion",
    "seg_siamunet_conc", "seg_siamunet_diff", "seg_unet",
    "seg_linear_probe", "seg_siam_linear_probe",
    "cls_linear", "cls_linear_probe",
]

DATASET_RE = re.compile(
    r"_(" + "|".join(re.escape(d) for d in DATASET_ORDER) + r")_"
)

print("Config loaded.")

In [14]:
# ── Core parsing functions ───────────────────────────────────────────────────

def parse_run_dir_name(name: str) -> dict:
    """Extract dataset, encoder, decoder from a run directory name."""
    m = DATASET_RE.search(name)
    if not m:
        return {}
    dataset = m.group(1)
    rest = name[m.end():]

    encoder = next((e for e in KNOWN_ENCODERS if rest.startswith(e + "_")), None)
    if encoder:
        rest = rest[len(encoder) + 1:]

    decoder = next((d for d in KNOWN_DECODERS if rest.startswith(d)), None)

    return {"dataset": dataset, "encoder": encoder or "", "decoder": decoder or ""}


def parse_log_metrics(log_path: Path) -> dict:
    """
    Parse train.log-0 for test metrics.

    Extracts the Mean values from [test] IoU, F1, Precision, Recall sections.
    Returns a dict with keys: test_mIoU, test_mF1, test_mPrecision, test_mRecall.
    For change detection (IoU section contains exactly 2 classes), also returns test_IoU.
    """
    if not log_path.exists():
        return {}

    text = log_path.read_text(errors="ignore")

    # Find all [test] sections — there may be multiple if val was run at test time
    # We want the LAST occurrence (final test run)
    sections = re.split(r"\[test\] ------- IoU", text)
    if len(sections) < 2:
        return {}
    test_block = "[test] ------- IoU" + sections[-1]

    metrics = {}
    metric_map = {
        "IoU":       "test_mIoU",
        "F1-score":  "test_mF1",
        "Precision": "test_mPrecision",
        "Recall":    "test_mRecall",
    }

    for section_name, metric_key in metric_map.items():
        # Find the Mean line inside this section
        pattern = rf"\[test\] ------- {section_name}.*?\[test\] Mean\s+([\d.]+)"
        m = re.search(pattern, test_block, re.DOTALL)
        if m:
            metrics[metric_key] = float(m.group(1))

    return metrics


def load_experiment_folder(
    folder: str | Path,
    encoder_map: dict | None = None,
    dataset_map: dict | None = None,
    metric: str = "test_mIoU",
    keep_all_metrics: bool = False,
) -> pd.DataFrame:
    """
    Load all runs from an experiment folder and return a long-form DataFrame.

    Parameters
    ----------
    folder : path to an experiment directory (e.g. pangaea_full_seg/upernet/panopticon_seg)
             OR a parent directory containing multiple encoder subdirectories.
             The function auto-detects whether run dirs are direct children or
             one level deeper.
    encoder_map : optional override for encoder display names {raw: display}
    dataset_map : optional override for dataset display names {raw: display}
    metric : which metric column to extract (default: test_mIoU)
    keep_all_metrics : if True, keep all parsed metrics instead of just `metric`

    Returns
    -------
    Long-form DataFrame with columns: run_dir, encoder, decoder, dataset, [metrics...]
    """
    enc_map = {**ENCODER_LABELS, **(encoder_map or {})}
    ds_map  = {**DATASET_LABELS,  **(dataset_map  or {})}
    folder  = Path(folder)

    # Collect candidate run directories:
    # A run dir contains train.log-0. Walk up to 2 levels deep.
    run_dirs = []
    for child in sorted(folder.iterdir()):
        if not child.is_dir():
            continue
        if (child / "train.log-0").exists():
            run_dirs.append(child)
        else:
            for grandchild in sorted(child.iterdir()):
                if grandchild.is_dir() and (grandchild / "train.log-0").exists():
                    run_dirs.append(grandchild)

    rows = []
    for run_dir in run_dirs:
        parsed = parse_run_dir_name(run_dir.name)
        if not parsed.get("dataset"):
            continue
        log_metrics = parse_log_metrics(run_dir / "train.log-0")
        if not log_metrics:
            continue

        enc_raw = parsed["encoder"]
        ds_raw  = parsed["dataset"]

        row = {
            "run_dir": run_dir.name,
            "encoder_raw": enc_raw,
            "decoder":     parsed["decoder"],
            "dataset_raw": ds_raw,
            "encoder":     enc_map.get(enc_raw, enc_raw),
            "dataset":     ds_map.get(ds_raw, ds_raw),
        }
        if keep_all_metrics:
            row.update(log_metrics)
        elif metric in log_metrics:
            row[metric] = log_metrics[metric]
        else:
            continue
        rows.append(row)

    return pd.DataFrame(rows)


def to_pivot(
    df: pd.DataFrame,
    metric: str = "test_mIoU",
    index: str = "encoder",
    columns: str = "dataset",
    aggfunc: str = "max",
    dataset_order: list | None = None,
    encoder_order: list | None = None,
    add_mean: bool = True,
    add_avg_rank: bool = True,
) -> pd.DataFrame:
    """
    Pivot a long-form results DataFrame to encoder × dataset.

    Parameters
    ----------
    df           : output of load_experiment_folder() or load_geobench_seg_folder()
    metric       : value column to pivot
    index        : row grouping (default: encoder display name)
    columns      : column grouping (default: dataset display name)
    aggfunc      : how to aggregate duplicates — 'max', 'mean', or 'last'
    dataset_order: explicit column order; defaults to DATASET_LABELS order
    encoder_order: explicit row order
    add_mean     : whether to append a Mean column (mean mIoU over available tasks)
    add_avg_rank : whether to append an Avg Rank column (mean per-task rank,
                   rank 1 = best; lower is better; NaN tasks are skipped)
    """
    if metric not in df.columns:
        raise ValueError(f"Metric '{metric}' not in DataFrame columns: {list(df.columns)}")

    pivot = df.pivot_table(
        index=index, columns=columns, values=metric,
        aggfunc=aggfunc
    )

    # Apply column order
    if dataset_order is None:
        ordered_ds = [DATASET_LABELS[d] for d in DATASET_ORDER if DATASET_LABELS[d] in pivot.columns]
        remaining  = [c for c in pivot.columns if c not in ordered_ds]
        dataset_order = ordered_ds + remaining
    pivot = pivot.reindex(columns=[c for c in dataset_order if c in pivot.columns])

    # Apply row order
    if encoder_order is not None:
        pivot = pivot.reindex([e for e in encoder_order if e in pivot.index])

    # Avg Rank: rank encoders within each dataset (1 = best), then average across tasks
    if add_avg_rank:
        data_cols = list(pivot.columns)
        ranks = pivot[data_cols].rank(axis=0, ascending=False, method="average", na_option="keep")
        pivot["Avg Rank"] = ranks.mean(axis=1, skipna=True).round(2)

    if add_mean:
        data_cols = [c for c in pivot.columns if c not in ("Mean", "Avg Rank")]
        pivot.insert(len(pivot.columns) - (1 if add_avg_rank else 0),
                     "Mean",
                     pivot[data_cols].mean(axis=1, skipna=True).round(3))

    return pivot.round(2)


print("Functions defined: load_experiment_folder(), to_pivot()")

Functions defined: load_experiment_folder(), to_pivot()


## Full-band UPerNet results

In [15]:
_VIT_FROZEN_MAP = {
    "vit_scratch":       "ViT-Frozen",
    "vit_scratch_mi":    "ViT-Frozen",
    "vit_scratch_p8":    "ViT-Frozen-p8",
    "vit_scratch_mi_p8": "ViT-Frozen-p8",
}

upernet_path = EXPERIMENTS_BASE / "pangaea_full_seg" / "upernet"
_dfs = []
for sf in sorted(upernet_path.iterdir()):
    if not sf.is_dir():
        continue
    enc_map = _VIT_FROZEN_MAP if "frozen" in sf.name else None
    _dfs.append(load_experiment_folder(sf, encoder_map=enc_map, metric="test_mIoU"))

df_upernet = pd.concat(_dfs, ignore_index=True)
print(f"{len(df_upernet)} runs loaded")
df_upernet.groupby(["encoder", "dataset"])["test_mIoU"].max().unstack()

100 runs loaded


dataset,AI4SmallFarms,CropTypeMapping,DynamicEarthNet,FiveBillionPixels,HLS-BurnScars,MADOS,PASTIS,Sen1Floods11,SpaceNet7-CD
encoder,,,,,,,,,
CARL,33.740,44.655,37.385,50.323,80.841,50.495,37.783,88.168,58.902
Copernicus-FM,33.346,48.085,37.715,62.801,82.332,56.897,33.785,89.110,62.360
DINOv2,35.205,36.729,26.583,64.580,74.482,58.777,18.777,70.020,62.845
DOFA,36.435,36.213,41.124,69.929,80.644,60.389,30.373,87.692,61.068
DOFAv2,38.084,36.231,42.941,71.216,78.274,48.602,19.792,88.008,63.634
Panopticon,37.883,47.994,40.553,70.096,84.225,68.217,36.814,90.591,60.769
SMARTIES,35.465,52.036,39.413,57.318,82.845,62.282,35.750,90.052,59.508
UNet,46.337,36.706,39.473,65.588,85.486,53.668,26.843,91.038,62.022
ViT-Frozen,32.618,45.055,31.559,48.336,80.838,44.230,6.584,88.077,57.639


In [16]:
pivot_upernet = to_pivot(df_upernet)
pivot_upernet

dataset,HLS-BurnScars,Sen1Floods11,MADOS,PASTIS,DynamicEarthNet,CropTypeMapping,SpaceNet7-CD,AI4SmallFarms,FiveBillionPixels,Mean,Avg Rank
encoder,,,,,,,,,,,
CARL,80.84,88.17,50.50,37.78,37.38,44.66,58.90,33.74,50.32,53.59,6.22
Copernicus-FM,82.33,89.11,56.90,33.78,37.72,48.08,62.36,33.35,62.80,56.27,4.78
DINOv2,74.48,70.02,58.78,18.78,26.58,36.73,62.84,35.20,64.58,49.78,7.44
DOFA,80.64,87.69,60.39,30.37,41.12,36.21,61.07,36.44,69.93,55.98,5.56
DOFAv2,78.27,88.01,48.60,19.79,42.94,36.23,63.63,38.08,71.22,54.09,5.44
Panopticon,84.22,90.59,68.22,36.81,40.55,47.99,60.77,37.88,70.10,59.68,2.67
SMARTIES,82.84,90.05,62.28,35.75,39.41,52.04,59.51,35.47,57.32,57.18,4.22
UNet,85.49,91.04,53.67,26.84,39.47,36.71,62.02,46.34,65.59,56.35,4.00
ViT-Frozen,80.84,88.08,44.23,6.58,31.56,45.06,57.64,32.62,48.34,48.33,8.89


## Full-band MLP Fusion results

In [17]:
mlp_path = EXPERIMENTS_BASE / "pangaea_full_seg" / "mlp_fusion"
_dfs = []
for sf in sorted(mlp_path.iterdir()):
    if not sf.is_dir():
        continue
    enc_map = _VIT_FROZEN_MAP if "frozen" in sf.name else None
    _dfs.append(load_experiment_folder(sf, encoder_map=enc_map, metric="test_mIoU"))

df_mlp = pd.concat(_dfs, ignore_index=True)
print(f"{len(df_mlp)} runs loaded")
pivot_mlp = to_pivot(df_mlp)
pivot_mlp

92 runs loaded


dataset,HLS-BurnScars,Sen1Floods11,MADOS,PASTIS,DynamicEarthNet,CropTypeMapping,SpaceNet7-CD,AI4SmallFarms,FiveBillionPixels,Mean,Avg Rank
encoder,,,,,,,,,,,
CARL,79.08,86.85,45.21,36.87,37.17,47.33,57.49,30.42,46.00,51.82,4.89
Copernicus-FM,80.74,85.06,49.66,29.85,35.75,47.03,57.40,34.68,59.12,53.26,4.78
DINOv2,74.10,68.18,57.28,13.94,23.51,19.80,58.27,34.20,62.35,45.74,7.00
DOFA,81.40,84.84,52.93,24.45,38.02,50.37,57.03,35.49,66.90,54.60,3.56
DOFAv2,74.84,84.24,44.31,19.78,42.30,22.70,57.03,35.15,66.86,49.69,5.89
Panopticon,83.38,85.88,66.75,34.48,35.82,43.50,58.50,34.41,66.22,56.55,2.89
SMARTIES,81.47,85.26,55.42,33.76,34.93,47.99,49.78,33.82,51.77,52.69,5.00
ViT-Frozen,79.83,84.65,32.96,26.58,33.47,49.88,49.78,32.40,41.88,47.94,7.44
ViT-Frozen-p8,78.81,87.12,39.53,33.79,33.61,49.33,50.54,32.18,42.01,49.66,6.11


## RGB UPerNet results

In [ ]:
df_rgb_up = load_experiment_folder(
    EXPERIMENTS_BASE / "pangaea_rgb" / "upernet",
    metric="test_mIoU",
)

# DINOv2 and DINOv3 are RGB-only — pull their results from the full-seg folder
_dinov2_up = load_experiment_folder(
    EXPERIMENTS_BASE / "pangaea_full_seg" / "upernet" / "dinov2_seg",
    metric="test_mIoU",
)
_dinov3_up = load_experiment_folder(
    EXPERIMENTS_BASE / "pangaea_full_seg" / "upernet" / "dinov3_vitl_seg",
    metric="test_mIoU",
)
df_rgb_up = pd.concat([df_rgb_up, _dinov2_up, _dinov3_up], ignore_index=True)

print(f"{len(df_rgb_up)} runs loaded")
pivot_rgb_up = to_pivot(df_rgb_up)
pivot_rgb_up.drop(columns=["SpaceNet7-CD"], inplace=True)
pivot_rgb_up

## RGB MLP Fusion results

In [ ]:
df_rgb_mlp = load_experiment_folder(
    EXPERIMENTS_BASE / "pangaea_rgb" / "mlp_fusion",
    metric="test_mIoU",
)

# DINOv2 and DINOv3 are RGB-only — pull their results from the full-seg folder
_dinov2_mlp = load_experiment_folder(
    EXPERIMENTS_BASE / "pangaea_full_seg" / "mlp_fusion" / "dinov2_mlp_fusion",
    metric="test_mIoU",
)
_dinov3_mlp = load_experiment_folder(
    EXPERIMENTS_BASE / "pangaea_full_seg" / "mlp_fusion" / "dinov3_vitl_mlp_fusion",
    metric="test_mIoU",
)
df_rgb_mlp = pd.concat([df_rgb_mlp, _dinov2_mlp, _dinov3_mlp], ignore_index=True)

print(f"{len(df_rgb_mlp)} runs loaded")
pivot_rgb_mlp = to_pivot(df_rgb_mlp)
pivot_rgb_mlp.drop(columns=["SpaceNet7-CD"], inplace=True)
pivot_rgb_mlp

## Advanced usage examples

In [20]:
# Load a single encoder folder
df_panopticon = load_experiment_folder(
    EXPERIMENTS_BASE / "pangaea_full_seg" / "upernet" / "panopticon_seg",
    keep_all_metrics=True,  # get mIoU, mF1, Precision, Recall
)
df_panopticon[["dataset", "test_mIoU", "test_mF1", "test_mPrecision", "test_mRecall"]]

,dataset,test_mIoU,test_mF1,test_mPrecision,test_mRecall
0,DynamicEarthNet,40.553,51.097,51.891,54.459
1,PASTIS,36.814,50.169,46.795,58.957
2,FiveBillionPixels,70.096,80.318,79.304,82.316
3,MADOS,68.217,78.297,78.187,80.500
4,Sen1Floods11,90.591,94.924,94.257,95.619
5,CropTypeMapping,47.994,60.619,65.871,66.839
6,HLS-BurnScars,84.225,90.937,90.860,91.013
7,AI4SmallFarms,37.883,53.821,58.305,56.179
8,SpaceNet7-CD,60.769,68.017,72.581,65.009


In [21]:
# Compare UPerNet vs MLP fusion side-by-side for full-band runs
mean_up  = to_pivot(df_upernet)[["Mean"]].rename(columns={"Mean": "UPerNet"})
mean_mlp = to_pivot(df_mlp)[["Mean"]].rename(columns={"Mean": "MLP Fusion"})
comparison = mean_up.join(mean_mlp, how="outer")
comparison["Δ (UPerNet - MLP)"] = (comparison["UPerNet"] - comparison["MLP Fusion"]).round(2)
comparison.sort_values("UPerNet", ascending=False)

dataset,UPerNet,MLP Fusion,Δ (UPerNet - MLP)
encoder,,,
Panopticon,59.68,56.55,3.13
SMARTIES,57.18,52.69,4.49
UNet,56.35,NaN,NaN
Copernicus-FM,56.27,53.26,3.01
DOFA,55.98,54.60,1.38
DOFAv2,54.09,49.69,4.40
CARL,53.59,51.82,1.77
ViT-Scratch,51.58,47.14,4.44
DINOv2,49.78,45.74,4.04


## GeoBench Segmentation results

In [ ]:
# ── GeoBench-specific constants ──────────────────────────────────────────────

GEOBENCH_SEG_DATASET_LABELS = {
    "mcashew-plantation":    "Cashew-Plant",
    "mchesapeake-landcover": "Chesapeake",
    "mneontree":             "NeonTree",
    "mnz-cattle":            "NZ-Cattle",
    "mpv4ger-seg":           "PV4Ger-Seg",
    "msa-crop-type":         "SA-Crop-Type",
}
GEOBENCH_SEG_DATASET_ORDER = list(GEOBENCH_SEG_DATASET_LABELS.keys())

GEOBENCH_DATASET_RE = re.compile(
    r"_(" + "|".join(re.escape(d) for d in GEOBENCH_SEG_DATASET_ORDER) + r")_"
)

# Keyed by experiment subfolder name so vit_frozen_p16 and vit_scratch are distinct
# even though both use the vit_scratch encoder config inside their run dirs.
GEOBENCH_FOLDER_ENCODER_LABELS = {
    "dofa":               "DOFA",
    "dofav2":             "DOFAv2",
    "carl":               "CARL",
    "carl_imagewise":     "CARL-Imagewise",
    "panopticon":         "Panopticon",
    "smarties_vitb":      "SMARTIES",
    "copernicus_fm_base": "Copernicus-FM",
    "dinov2":             "DINOv2",
    "dinov3_vitl":        "DINOv3-ViT-L",
    "unet":               "UNet",
    "vit_scratch":        "ViT-Scratch",
    "vit_frozen_p16":     "ViT-Frozen-p16",
    "vit_frozen_p8":      "ViT-Frozen-p8",
}

GEOBENCH_ENCODER_ORDER = [
    "DOFA", "DOFAv2", "CARL", "CARL-Imagewise", "Panopticon",
    "SMARTIES", "Copernicus-FM", "DINOv2", "DINOv3-ViT-L",
    "UNet", "ViT-Scratch", "ViT-Frozen-p16", "ViT-Frozen-p8",
]

GEOBENCH_DATASET_DISPLAY_ORDER = [
    GEOBENCH_SEG_DATASET_LABELS[d] for d in GEOBENCH_SEG_DATASET_ORDER
]


def load_geobench_seg_folder(base_path, metric="test_mIoU"):
    """
    Load GeoBench seg results from base_path/<enc_folder>/<run_dir>/train.log-0.

    Uses the subfolder name (enc_folder) for the encoder display label so that
    vit_frozen_p16 and vit_scratch are distinguished even though both contain
    runs with the same vit_scratch encoder config string in their directory names.
    """
    rows = []
    for enc_folder in sorted(Path(base_path).iterdir()):
        if not enc_folder.is_dir():
            continue
        enc_label = GEOBENCH_FOLDER_ENCODER_LABELS.get(enc_folder.name, enc_folder.name)

        for run_dir in sorted(enc_folder.iterdir()):
            if not run_dir.is_dir():
                continue
            m = GEOBENCH_DATASET_RE.search(run_dir.name)
            if not m:
                continue
            ds_raw = m.group(1)
            log_metrics = parse_log_metrics(run_dir / "train.log-0")
            if metric not in log_metrics:
                continue
            rows.append({
                "encoder":     enc_label,
                "dataset":     GEOBENCH_SEG_DATASET_LABELS.get(ds_raw, ds_raw),
                "dataset_raw": ds_raw,
                metric:        log_metrics[metric],
            })
    return pd.DataFrame(rows)


print("GeoBench constants and load_geobench_seg_folder() defined.")

In [23]:
# GeoBench — UPerNet
df_geo_up = load_geobench_seg_folder(
    EXPERIMENTS_BASE / "geobench" / "seg" / "UPerNet",
    metric="test_mIoU",
)
print(f"{len(df_geo_up)} runs loaded")

pivot_geo_up = to_pivot(
    df_geo_up,
    metric="test_mIoU",
    dataset_order=GEOBENCH_DATASET_DISPLAY_ORDER,
    encoder_order=GEOBENCH_ENCODER_ORDER,
)
pivot_geo_up

66 runs loaded


dataset,Cashew-Plant,Chesapeake,NeonTree,NZ-Cattle,PV4Ger-Seg,SA-Crop-Type,Mean,Avg Rank
encoder,,,,,,,,
DOFA,66.90,59.63,58.67,84.36,94.93,32.44,66.16,4.50
DOFAv2,65.20,61.00,61.56,84.89,95.50,29.78,66.32,4.33
CARL,61.73,51.67,54.43,80.74,88.70,31.56,61.47,9.83
CARL-Imagewise,64.89,51.85,54.18,81.08,88.11,33.04,62.19,8.83
Panopticon,73.17,60.11,55.58,84.14,94.19,36.21,67.23,3.67
SMARTIES,71.06,61.22,54.39,82.48,91.54,32.70,65.56,6.00
Copernicus-FM,69.90,66.52,56.52,84.07,95.09,34.04,67.69,3.50
UNet,80.78,71.45,57.32,85.00,93.85,34.50,70.48,2.17
ViT-Scratch,75.05,56.68,54.65,82.90,92.70,33.85,65.97,5.17


In [24]:
# GeoBench — MLP Fusion
df_geo_mlp = load_geobench_seg_folder(
    EXPERIMENTS_BASE / "geobench" / "seg" / "MLPFusion",
    metric="test_mIoU",
)
print(f"{len(df_geo_mlp)} runs loaded")

pivot_geo_mlp = to_pivot(
    df_geo_mlp,
    metric="test_mIoU",
    dataset_order=GEOBENCH_DATASET_DISPLAY_ORDER,
    encoder_order=GEOBENCH_ENCODER_ORDER,
)
pivot_geo_mlp

60 runs loaded


dataset,Cashew-Plant,Chesapeake,NeonTree,NZ-Cattle,PV4Ger-Seg,SA-Crop-Type,Mean,Avg Rank
encoder,,,,,,,,
DOFA,24.48,57.79,56.89,66.59,92.68,31.14,54.93,5.33
DOFAv2,26.89,67.00,59.75,72.48,94.02,28.69,58.14,4.00
CARL,26.36,49.55,50.32,77.29,87.20,31.08,53.63,6.67
CARL-Imagewise,30.38,50.02,50.94,77.43,86.16,32.18,54.52,4.83
Panopticon,31.94,64.16,51.60,73.57,92.91,34.55,58.12,2.33
SMARTIES,29.25,58.53,50.81,66.39,88.75,33.68,54.57,4.83
Copernicus-FM,28.51,62.34,54.37,68.49,92.87,34.11,56.78,3.50
ViT-Scratch,20.91,47.91,48.97,62.15,85.99,29.21,49.19,9.17
ViT-Frozen-p16,21.32,50.73,48.97,62.18,85.91,29.14,49.71,8.83


## Decoder Sensitivity Analysis

How much does the choice of decoder (UPerNet vs MLP Fusion) change the relative ranking of encoders?

**Colour convention (consistent across both plots):**
- **Blue** = encoder ranks better with **UPerNet**
- **Red** = encoder ranks better with **MLP Fusion**

- **Plot 1 — Rank scatter**: UPerNet avg rank vs MLP Fusion avg rank, one point per encoder. Points on the diagonal = decoder has no effect. Point colour and background shading follow the blue/red convention.
- **Plot 2 — Δ rank heatmap**: `rank_UPerNet − rank_MLP_Fusion` per encoder × dataset. Negative/blue = UPerNet better. Positive/red = MLP Fusion better. Rightmost column = mean |Δ rank| (sensitivity score).

Ranks are recomputed on the **common encoder set** (UNet excluded — it has no MLP Fusion counterpart).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns


def compute_decoder_sensitivity(pivot_up, pivot_mlp, dataset_order=None):
    """
    Recompute per-dataset ranks on the common encoder set (fair comparison).

    delta = rank_UP − rank_MLP:
      negative (blue) = encoder ranks better (lower rank number) under UPerNet
      positive  (red) = encoder ranks better under MLP Fusion
    """
    skip = {"Mean", "Avg Rank"}
    cols_up  = [c for c in pivot_up.columns  if c not in skip]
    cols_mlp = [c for c in pivot_mlp.columns if c not in skip]

    common_datasets = [c for c in cols_up if c in cols_mlp]
    if dataset_order:
        common_datasets = [d for d in dataset_order if d in common_datasets]

    common_encoders = [e for e in pivot_up.index if e in pivot_mlp.index]

    up  = pivot_up.loc[common_encoders, common_datasets]
    mlp = pivot_mlp.loc[common_encoders, common_datasets]

    rank_up  = up.rank(axis=0, ascending=False, method="average", na_option="keep")
    rank_mlp = mlp.rank(axis=0, ascending=False, method="average", na_option="keep")

    delta          = (rank_up - rank_mlp).round(1)   # neg = UPerNet better, pos = MLP better
    avg_rank_up    = rank_up.mean(axis=1,  skipna=True).round(2)
    avg_rank_mlp   = rank_mlp.mean(axis=1, skipna=True).round(2)
    mean_abs_delta = delta.abs().mean(axis=1).round(2)

    return delta, avg_rank_up, avg_rank_mlp, mean_abs_delta


def plot_decoder_sensitivity(pivot_up, pivot_mlp, title,
                              dataset_order=None, encoder_order=None):
    """
    Produces two separate figures with a shared colour convention:
      blue = encoder benefits from UPerNet
      red  = encoder benefits from MLP Fusion

    Colormap used in both figures: RdBu_r (reversed RdBu)
      negative delta → blue  (UPerNet better)
      positive delta → red   (MLP Fusion better)

    Figure 1 (Rank Scatter):
      X = UPerNet avg rank, Y = MLP Fusion avg rank (1 = best).
      Point colour = avg_rank_UP − avg_rank_MLP (RdBu_r, centred at 0).
      Points above diagonal = UPerNet better = blue.
      Points below diagonal = MLP better = red.

    Figure 2 (Δ Rank Heatmap):
      Cells = rank_UP − rank_MLP per encoder × dataset (RdBu_r, centred at 0).
      Rightmost column = mean |Δ rank| per encoder (YlOrRd magnitude scale).
    """
    delta, avg_rank_up, avg_rank_mlp, mean_abs_delta = compute_decoder_sensitivity(
        pivot_up, pivot_mlp, dataset_order=dataset_order
    )
    if encoder_order:
        order = [e for e in encoder_order if e in delta.index]
        delta          = delta.loc[order]
        avg_rank_up    = avg_rank_up[order]
        avg_rank_mlp   = avg_rank_mlp[order]
        mean_abs_delta = mean_abs_delta[order]

    n_enc, n_ds = delta.shape

    # Signed colour value: neg (blue) = UPerNet better, pos (red) = MLP better
    signed_diff = avg_rank_up - avg_rank_mlp
    sym = max(float(signed_diff.abs().max()), 0.5)

    # ── Figure 1: Rank scatter ────────────────────────────────────────────────
    fig1, ax1 = plt.subplots(figsize=(6.5, 6.5))

    lo = max(0.3, min(float(avg_rank_up.min()), float(avg_rank_mlp.min())) - 0.5)
    hi = max(float(avg_rank_up.max()), float(avg_rank_mlp.max())) + 1.2

    ax1.fill([lo, lo, hi], [lo, hi, hi], color="steelblue", alpha=0.07, zorder=0)
    ax1.fill([lo, hi, hi], [lo, lo, hi], color="firebrick", alpha=0.07, zorder=0)
    ax1.plot([lo, hi], [lo, hi], "--", color="0.5", lw=1, zorder=1)

    sc = ax1.scatter(
        avg_rank_up, avg_rank_mlp,
        c=signed_diff, cmap="RdBu_r", vmin=-sym, vmax=sym,
        s=160, zorder=3, edgecolors="0.3", linewidths=0.7,
    )
    fig1.colorbar(sc, ax=ax1,
                  label="avg_rank_UP − avg_rank_MLP  (blue → UPerNet better, red → MLP better)",
                  shrink=0.78, pad=0.02)

    for enc in avg_rank_up.index:
        ax1.annotate(enc, (float(avg_rank_up[enc]), float(avg_rank_mlp[enc])),
                     xytext=(6, 4), textcoords="offset points", fontsize=8.5)

    ax1.text(lo + 0.1, hi - 0.15, "UPerNet better →", fontsize=8,
             color="steelblue", va="top", fontstyle="italic")
    ax1.text(hi - 0.1, lo + 0.15, "← MLP Fusion better", fontsize=8,
             color="firebrick", ha="right", va="bottom", fontstyle="italic")

    ax1.set_xlabel("Avg Rank — UPerNet  (1 = best)")
    ax1.set_ylabel("Avg Rank — MLP Fusion  (1 = best)")
    ax1.set_title(f"{title}\nRank Scatter")
    ax1.set_xlim(lo, hi)
    ax1.set_ylim(lo, hi)
    ax1.set_aspect("equal")
    fig1.tight_layout()
    plt.show()

    # ── Figure 2: Δ rank heatmap ──────────────────────────────────────────────
    fig2, ax2 = plt.subplots(figsize=(2.5 + n_ds * 0.9, max(4, n_enc * 0.55 + 2.0)))

    vmax = max(1.0, float(delta.abs().max().max()))

    sns.heatmap(
        delta, ax=ax2,
        cmap="RdBu_r", center=0, vmin=-vmax, vmax=vmax,
        annot=True, fmt=".1f", annot_kws={"size": 8.5},
        linewidths=0.4, linecolor="white",
        cbar_kws={"label": "Δ rank  (−→ UPerNet better,  +→ MLP better)", "shrink": 0.78},
        yticklabels=True,
    )

    # Mean |Δ| column appended to the right
    ax2.set_xlim(0, n_ds + 1.7)
    ax2.axvline(n_ds, color="0.4", lw=1.5, ls=":")
    ax2.text(n_ds + 0.85, -0.5, "Mean\n|Δ|", ha="center", va="bottom",
             fontsize=8, fontweight="bold")

    max_val = float(mean_abs_delta.max()) or 1.0
    for i, enc in enumerate(delta.index):
        val = float(mean_abs_delta[enc])
        clr = plt.cm.YlOrRd(val / max_val)
        rect = mpatches.Rectangle((n_ds + 0.1, i + 0.1), 1.5, 0.8,
                                   color=clr, zorder=2)
        ax2.add_patch(rect)
        ax2.text(n_ds + 0.85, i + 0.5, f"{val:.2f}",
                 ha="center", va="center", fontsize=8.5, fontweight="bold",
                 color="white" if val > 0.65 * max_val else "black", zorder=3)

    ax2.set_title(f"{title}\nΔ Rank Heatmap")
    ax2.tick_params(axis="x", labelrotation=40)
    ax2.set_ylabel("")
    fig2.tight_layout()
    plt.show()

    return delta, avg_rank_up, avg_rank_mlp, mean_abs_delta


print("compute_decoder_sensitivity() and plot_decoder_sensitivity() defined.")

In [ ]:
def plot_miou_delta_heatmap(pivot_up, pivot_mlp, title,
                             dataset_order=None, encoder_order=None):
    """
    Heatmap of absolute mIoU difference per encoder × dataset.

    delta = mIoU_MLP − mIoU_UP  (same sign convention as rank heatmap):
      negative / blue = UPerNet gives higher mIoU
      positive / red  = MLP Fusion gives higher mIoU

    Uses cmap='RdBu_r' (negative → blue) to match the rank heatmap.
    Rightmost column = mean |Δ mIoU| per encoder (sensitivity magnitude).
    """
    skip = {"Mean", "Avg Rank"}
    cols_up  = [c for c in pivot_up.columns  if c not in skip]
    cols_mlp = [c for c in pivot_mlp.columns if c not in skip]

    common_datasets = [c for c in cols_up if c in cols_mlp]
    if dataset_order:
        common_datasets = [d for d in dataset_order if d in common_datasets]

    common_encoders = [e for e in pivot_up.index if e in pivot_mlp.index]
    if encoder_order:
        common_encoders = [e for e in encoder_order if e in common_encoders]

    up  = pivot_up.loc[common_encoders, common_datasets]
    mlp = pivot_mlp.loc[common_encoders, common_datasets]

    delta          = (mlp - up).round(1)     # negative = UPerNet better (matches rank heatmap)
    mean_abs_delta = delta.abs().mean(axis=1).round(2)

    n_enc, n_ds = delta.shape
    fig, ax = plt.subplots(figsize=(2.5 + n_ds * 0.9, max(4, n_enc * 0.55 + 2.0)))

    vmax = max(1.0, float(delta.abs().max().max()))

    sns.heatmap(
        delta, ax=ax,
        cmap="RdBu_r", center=0, vmin=-vmax, vmax=vmax,
        annot=True, fmt=".1f", annot_kws={"size": 8.5},
        linewidths=0.4, linecolor="white",
        cbar_kws={"label": "Δ mIoU  (−→ UPerNet better,  +→ MLP better)", "shrink": 0.78},
        yticklabels=True,
    )

    # Mean |Δ mIoU| column appended to the right
    ax.set_xlim(0, n_ds + 1.7)
    ax.axvline(n_ds, color="0.4", lw=1.5, ls=":")
    ax.text(n_ds + 0.85, -0.5, "Mean\n|Δ|", ha="center", va="bottom",
            fontsize=8, fontweight="bold")

    max_val = float(mean_abs_delta.max()) or 1.0
    for i, enc in enumerate(common_encoders):
        val = float(mean_abs_delta[enc])
        clr = plt.cm.YlOrRd(val / max_val)
        rect = mpatches.Rectangle((n_ds + 0.1, i + 0.1), 1.5, 0.8,
                                   color=clr, zorder=2)
        ax.add_patch(rect)
        ax.text(n_ds + 0.85, i + 0.5, f"{val:.2f}",
                ha="center", va="center", fontsize=8.5, fontweight="bold",
                color="white" if val > 0.65 * max_val else "black", zorder=3)

    ax.set_title(f"{title}\nΔ mIoU Heatmap  (MLP Fusion − UPerNet)")
    ax.tick_params(axis="x", labelrotation=40)
    ax.set_ylabel("")
    fig.tight_layout()
    plt.show()

    return delta, mean_abs_delta


def plot_rank_slopegraph(pivot_up, pivot_mlp, title,
                          dataset_order=None, encoder_order=None):
    """
    Slopegraph showing how each encoder's overall rank changes between decoders.

    Left  = UPerNet ranking  (rank 1 = best, at top).
    Right = MLP Fusion ranking.
    Lines coloured blue (UPerNet better) or red (MLP Fusion better).
    Avg rank across datasets is shown in parentheses next to each name.
    """
    from matplotlib.lines import Line2D

    _, avg_rank_up, avg_rank_mlp, _ = compute_decoder_sensitivity(
        pivot_up, pivot_mlp, dataset_order=dataset_order
    )
    if encoder_order:
        order = [e for e in encoder_order if e in avg_rank_up.index]
    else:
        order = list(avg_rank_up.index)

    avg_rank_up  = avg_rank_up[order]
    avg_rank_mlp = avg_rank_mlp[order]

    # Sort each side by its own avg rank → determines vertical position
    left_order  = sorted(order, key=lambda e: avg_rank_up[e])
    right_order = sorted(order, key=lambda e: avg_rank_mlp[e])

    left_pos  = {enc: i for i, enc in enumerate(left_order)}
    right_pos = {enc: i for i, enc in enumerate(right_order)}

    n = len(order)
    fig, ax = plt.subplots(figsize=(9, max(5, n * 0.78 + 1.8)))

    for enc in order:
        lp = left_pos[enc]
        rp = right_pos[enc]

        # lp < rp: encoder has a better position (lower index) on the left → UPerNet better
        if lp < rp:
            color = "steelblue"
        elif lp > rp:
            color = "firebrick"
        else:
            color = "0.55"

        ax.plot([0, 1], [lp, rp], color=color, lw=2.5, alpha=0.8,
                solid_capstyle="round")
        ax.scatter([0, 1], [lp, rp], color=color, s=90, zorder=5)

        ax.text(-0.05, lp,
                f"#{lp + 1}  {enc}  ({avg_rank_up[enc]:.1f})",
                ha="right", va="center", fontsize=10)
        ax.text(1.05, rp,
                f"({avg_rank_mlp[enc]:.1f})  {enc}  #{rp + 1}",
                ha="left", va="center", fontsize=10)

    # Vertical axis lines
    ax.axvline(0, color="0.3", lw=1.5)
    ax.axvline(1, color="0.3", lw=1.5)

    # Decoder headers
    ax.text(0, -1.1, "UPerNet", ha="center", fontweight="bold", fontsize=12)
    ax.text(1, -1.1, "MLP Fusion", ha="center", fontweight="bold", fontsize=12)

    # Legend
    legend_elements = [
        Line2D([0], [0], color="steelblue", lw=2.5, label="UPerNet better"),
        Line2D([0], [0], color="firebrick", lw=2.5, label="MLP Fusion better"),
        Line2D([0], [0], color="0.55",      lw=2.5, label="No change"),
    ]
    ax.legend(handles=legend_elements, loc="upper center",
              bbox_to_anchor=(0.5, -0.04), ncol=3, frameon=False, fontsize=10)

    ax.set_xlim(-0.9, 1.9)
    ax.set_ylim(-1.5, n - 0.5)
    ax.invert_yaxis()    # rank 1 at top
    ax.axis("off")
    ax.set_title(title, fontsize=13, pad=15)
    fig.tight_layout()
    plt.show()


print("plot_miou_delta_heatmap() and plot_rank_slopegraph() defined.")

### Pangaea Benchmark

In [ ]:
pangaea_ds_order = [DATASET_LABELS[d] for d in DATASET_ORDER]

delta_pangaea, _, _, _ = plot_decoder_sensitivity(
    pivot_upernet, pivot_mlp,
    title="Pangaea Benchmark — Decoder Sensitivity (UPerNet vs MLP Fusion)",
    dataset_order=pangaea_ds_order,
)

In [ ]:
plot_miou_delta_heatmap(
    pivot_upernet, pivot_mlp,
    title="Pangaea Benchmark — Decoder Sensitivity (UPerNet vs MLP Fusion)",
    dataset_order=pangaea_ds_order,
)

In [ ]:
plot_rank_slopegraph(
    pivot_upernet, pivot_mlp,
    title="Pangaea Benchmark — Rank Change: UPerNet vs MLP Fusion",
    dataset_order=pangaea_ds_order,
)

### GeoBench

In [ ]:
delta_geo, _, _, _ = plot_decoder_sensitivity(
    pivot_geo_up, pivot_geo_mlp,
    title="GeoBench — Decoder Sensitivity (UPerNet vs MLP Fusion)",
    dataset_order=GEOBENCH_DATASET_DISPLAY_ORDER,
    encoder_order=GEOBENCH_ENCODER_ORDER,
)

In [ ]:
plot_miou_delta_heatmap(
    pivot_geo_up, pivot_geo_mlp,
    title="GeoBench — Decoder Sensitivity (UPerNet vs MLP Fusion)",
    dataset_order=GEOBENCH_DATASET_DISPLAY_ORDER,
    encoder_order=GEOBENCH_ENCODER_ORDER,
)

In [ ]:
plot_rank_slopegraph(
    pivot_geo_up, pivot_geo_mlp,
    title="GeoBench — Rank Change: UPerNet vs MLP Fusion",
    dataset_order=GEOBENCH_DATASET_DISPLAY_ORDER,
    encoder_order=GEOBENCH_ENCODER_ORDER,
)

## GeoBench Classification results

Linear probe (`cls_linear` decoder, `finetune=False`) on 6 GeoBench classification datasets.

**Primary metric per dataset:**
- **BigEarthNet** (43-class multi-label) → macro **F1-score** (exact-match Accuracy is near zero for multi-label)
- All other datasets (single-label) → **Accuracy**

Results stored in a unified `metric` column so the pivot table is directly comparable.

In [ ]:
# ── GeoBench Classification constants ────────────────────────────────────────

GEOBENCH_CLS_DATASET_LABELS = {
    "mbigearthnet": "BigEarthNet",
    "mbrickkiln":   "BrickKiln",
    "meurosat":     "EuroSAT",
    "mforestnet":   "ForestNet",
    "mpv4ger":      "PV4Ger",
    "mso2sat":      "So2Sat",
}
GEOBENCH_CLS_DATASET_ORDER = list(GEOBENCH_CLS_DATASET_LABELS.keys())

GEOBENCH_CLS_DATASET_RE = re.compile(
    r"_(" + "|".join(re.escape(d) for d in GEOBENCH_CLS_DATASET_ORDER) + r")_"
)

# Keyed by experiment subfolder name (same pattern as GeoBench seg)
GEOBENCH_CLS_FOLDER_ENCODER_LABELS = {
    "dofa":               "DOFA",
    "dofav2":             "DOFAv2",
    "carl":               "CARL",
    "carl_imagewise":     "CARL-Imagewise",
    "panopticon":         "Panopticon",
    "smarties_vitb":      "SMARTIES",
    "copernicus_fm_base": "Copernicus-FM",
    "dinov2":             "DINOv2",
    "dinov3_vitl":        "DINOv3-ViT-L",
    "vit_scratch":        "ViT-Scratch",
    "vit_frozen_p16":     "ViT-Frozen-p16",
    "vit_frozen_p8":      "ViT-Frozen-p8",
}

GEOBENCH_CLS_ENCODER_ORDER = [
    "DOFA", "DOFAv2", "CARL", "CARL-Imagewise", "Panopticon",
    "SMARTIES", "Copernicus-FM", "DINOv2", "DINOv3-ViT-L",
    "ViT-Scratch", "ViT-Frozen-p16", "ViT-Frozen-p8",
]

GEOBENCH_CLS_DATASET_DISPLAY_ORDER = [
    GEOBENCH_CLS_DATASET_LABELS[d] for d in GEOBENCH_CLS_DATASET_ORDER
]


def parse_cls_log_metrics(log_path: Path) -> dict:
    """
    Parse classification train.log-0 for test metrics (Accuracy, F1, Precision, Recall).
    Uses the LAST [test] block if multiple are present.
    """
    if not log_path.exists():
        return {}
    text = log_path.read_text(errors="ignore")
    sections = re.split(r"\[test\] ------- Accuracy", text)
    if len(sections) < 2:
        return {}
    test_block = "[test] ------- Accuracy" + sections[-1]
    metrics = {}
    for section_name, key in [
        ("Accuracy",  "test_Accuracy"),
        ("F1-score",  "test_mF1"),
        ("Precision", "test_Precision"),
        ("Recall",    "test_Recall"),
    ]:
        m = re.search(
            rf"\[test\] ------- {section_name}.*?\[test\] Mean\s+([\d.]+)",
            test_block, re.DOTALL,
        )
        if m:
            metrics[key] = float(m.group(1))
    return metrics


def load_geobench_cls_folder(base_path) -> pd.DataFrame:
    """
    Load GeoBench classification results from base_path/<enc_folder>/<run_dir>/train.log-0.

    Primary metric: Accuracy for single-label tasks; F1-score for mBigEarthNet (multi-label).
    All values stored in a unified 'metric' column.
    """
    rows = []
    for enc_folder in sorted(Path(base_path).iterdir()):
        if not enc_folder.is_dir():
            continue
        enc_label = GEOBENCH_CLS_FOLDER_ENCODER_LABELS.get(enc_folder.name, enc_folder.name)

        for run_dir in sorted(enc_folder.iterdir()):
            if not run_dir.is_dir():
                continue
            m = GEOBENCH_CLS_DATASET_RE.search(run_dir.name)
            if not m:
                continue
            ds_raw = m.group(1)
            log_metrics = parse_cls_log_metrics(run_dir / "train.log-0")
            if not log_metrics:
                continue

            metric_key = "test_mF1" if ds_raw == "mbigearthnet" else "test_Accuracy"
            value = log_metrics.get(metric_key)
            if value is None:
                continue

            rows.append({
                "encoder":     enc_label,
                "dataset":     GEOBENCH_CLS_DATASET_LABELS.get(ds_raw, ds_raw),
                "dataset_raw": ds_raw,
                "metric":      value,
            })
    return pd.DataFrame(rows)


print("GeoBench classification constants and functions defined.")
print("Primary metric: Accuracy (single-label) / F1-score (BigEarthNet multi-label)")

In [ ]:
df_geo_cls = load_geobench_cls_folder(
    EXPERIMENTS_BASE / "geobench" / "cls"
)
print(f"{len(df_geo_cls)} runs loaded")

pivot_geo_cls = to_pivot(
    df_geo_cls,
    metric="metric",
    dataset_order=GEOBENCH_CLS_DATASET_DISPLAY_ORDER,
    encoder_order=GEOBENCH_CLS_ENCODER_ORDER,
)
pivot_geo_cls